![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Building the Core RAG-Pipeline

*Project 3 — AI Product: From Model to Product*

**Team:** Zoé Doucet

**Track:** LLM

**Data:** Forest Entomology and Pathology

---
The objective is to build a Retrieval-Augmented Generation (RAG) pipeline for the Forest Pest Assistant.

The pipeline will retrieve relevant textbook passages using semantic embeddings and generate evidence-based answers with PDF page citations.

The TF-IDF baseline achieved a page-level Recall@5 of 25%. Semantic retrieval will be evaluated against this baseline to determine whether it improves the retrieval of relevant evidence.

The pipeline should also handle insufficient evidence without inventing diagnoses or treatment recommendations.

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sentence_transformers
from pypdf import PdfReader
from src.functions import clean_text, chunk_document
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

from src.functions import *
from src.functions import search_chroma

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

## 1.Data preparation

The cleaned textbook passages from Notebook 01 are reused to build the RAG pipeline.

The same cleaning and chunking settings are maintained to ensure consistency with the TF-IDF baseline. Each chunk retains its original PDF page number for source citations.

In [14]:
# Loading the book
pdf = PdfReader("../data/raw/Forest_Entomology_and_Pathology.pdf")

# Extracting text and excluding the first 12 PDF pages
searchable_pages = [
    {
        "pdf_page": i + 1,
        "text": clean_text(page.extract_text() or "")
    }
    for i, page in enumerate(pdf.pages)
    if i >= 12]

# Verifying the number of searchable pages
print("Searchable pages:", len(searchable_pages))
print("First searchable PDF page:", searchable_pages[0]["pdf_page"])

Searchable pages: 798
First searchable PDF page: 13


In [40]:
# Converting the list of pages into a DataFrame
pages_df = pd.DataFrame(searchable_pages)

# Using the column name expected by chunk_document()
pages_df = pages_df.rename(columns={"text": "clean_text"})

# Creating the chunks using the original settings
chunks = chunk_document(pages_df, size=150, overlap=30)

# Checking the result
print("Total chunks:", len(chunks))
print("First chunk ID:", chunks[0]["id"])
print("First chunk PDF page:", chunks[0]["pdf_page"])

Total chunks: 3274
First chunk ID: page_13_chunk_0
First chunk PDF page: 13


The chunks are ready for embedding and storage in the Chroma vector database. Maintaining the same chunks allows a consistent comparison between TF-IDF and semantic retrieval.

---
## 1. LLM track — the pipeline

Build it end to end, with the functions in `src/functions.py`:

- **Retrieval:** embed the chunks, store them (Chroma), retrieve the top `k` with metadata filters.
- **Generation:** a grounded prompt that answers only from the retrieved texts, cites their ids, and says "I don't know" otherwise.
- **Or an agent:** tools with clear descriptions (a retrieval tool, a calculator, web search), a step limit, and a trace you can read.

Run it on three or four questions from your evaluation set and read the answers in full.

#### 1.1 Embeddings and Chroma

Semantic embeddings convert textbook chunks into numerical vectors that represent their meaning.

The pretrained all-MiniLM-L6-v2 model produces 384-dimensional embeddings. Chroma will store these embeddings together with the chunk text and metadata, including PDF page numbers and chunk identifiers.

This allows the RAG pipeline to search for relevant evidence using semantic similarity.

In [10]:
# Creating a folder for the saved vector database
db_path = Path("../models/chroma")
db_path.mkdir(parents=True, exist_ok=True)

# Opening a persistent Chroma database
client = chromadb.PersistentClient(path=str(db_path))

# Creat or open the textbook collection
collection = client.get_or_create_collection(
    name="forest_textbook",
    metadata={"hnsw:space": "cosine"})

print("Chroma database ready")
print("Stored chunks:", collection.count())

Chroma database ready
Stored chunks: 3274


Reopen the Chroma datbase if it has already been created.

In [ ]:
# Reopening the saved Chroma database
client = chromadb.PersistentClient(path="../models/chroma")
collection = client.get_collection(name="forest_textbook")

print("Stored chunks:", collection.count())

Stored chunks: 3274


In [41]:
# Collecting the text from every chunk
texts = [chunk["text"] for chunk in chunks]

# Converting the text into numerical embeddings
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True)

# Checking the result
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (3274, 384)


In [42]:
# Preparing chunk IDs and PDF page metadata
ids = [chunk["id"] for chunk in chunks]
metadata = [
    {"pdf_page": chunk["pdf_page"]}
    for chunk in chunks
]

# Storing chunks in batches to avoid large database requests
batch_size = 500

for start in range(0, len(chunks), batch_size):
    end = start + batch_size

    collection.upsert(
        ids=ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadata[start:end])

# Checking the number of stored chunks
print("Stored chunks:", collection.count())

Stored chunks: 3274


As the page number of the book and the page number of the pdf file do not match, the page number of the printed book are added to the Chroma database.

In [43]:
# Adding printed book pages to existing Chroma metadata
batch_size = 500

for start in range(0, len(chunks), batch_size):
    batch = chunks[start:start + batch_size]

    batch_ids = [chunk["id"] for chunk in batch]

    batch_metadata = []

    for chunk in batch:
        pdf_page = chunk["pdf_page"]

        page_info = {"pdf_page": pdf_page}

        # Add a book page only when it was verified
        if pdf_page in book_pages:
            page_info["book_page"] = book_pages[pdf_page]

        batch_metadata.append(page_info)

    # Updating metadata without changing embeddings or documents
    collection.update(
        ids=batch_ids,
        metadatas=batch_metadata)

print("Updated metadata for", len(chunks), "chunks")
print("Stored chunks:", collection.count())

Updated metadata for 3274 chunks
Stored chunks: 3274


In [18]:
# verifying the Chroma database
# Retrieving the first chunk using its ID
stored = collection.get(
    ids=[chunks[0]["id"]],
    include=["documents", "metadatas"])

# Checking the stored information
print("Chunk ID:", stored["ids"][0])
print("PDF page:", stored["metadatas"][0]["pdf_page"])
print("Text preview:", stored["documents"][0][:200])

Chunk ID: page_13_chunk_0
PDF page: 13
Text preview: Chapter 1 Introduction to and Importance of Insects Richard Redak 1.1 Introduction Insects and closely related arthropods are the dominant and most diverse forms of terrestrial and aquatic (non-marine


The 3,274 textbook chunks were converted into 384-dimensional embeddings using the pretrained all-MiniLM-L6-v2 model.

The embeddings were stored in a persistent Chroma vector database, together with the original chunk texts, unique identifiers and PDF page numbers.

#### 1.2 Semantic retrieval

Semantic retrieval identifies textbook passages that are relevant to a question based on the similarity of their embeddings.

The initial retrieval setting is k=5, which will later be compared with another value in section 2.

In [ ]:
# Loading the embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [60]:
# Testing retrieval restricted to PDF page 326
results = search_chroma(
    question="What signs can indicate a bark beetle attack?",
    model=model,
    collection=collection,
    k=5,
    pdf_page=326)

for i in range(len(results["ids"][0])):
    print("Chunk:", results["ids"][0][i])
    print("PDF page:", results["metadatas"][0][i]["pdf_page"])
    print("Distance:", round(results["distances"][0][i], 3))
    print()

Chunk: page_326_chunk_0
PDF page: 326
Distance: 0.485

Chunk: page_326_chunk_240
PDF page: 326
Distance: 0.612

Chunk: page_326_chunk_360
PDF page: 326
Distance: 0.619

Chunk: page_326_chunk_480
PDF page: 326
Distance: 0.807

Chunk: page_326_chunk_120
PDF page: 326
Distance: 0.835



In [61]:
# Inspecting retrieved passages without a page filter
question = "What signs can indicate a bark beetle attack?"

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

for i in range(len(results["ids"][0])):
    print("PDF page:", results["metadatas"][0][i]["pdf_page"])
    print("Distance:", round(results["distances"][0][i], 3))
    print("Text:", results["documents"][0][i][:500])
    print("-" * 60)

PDF page: 771
Distance: 0.356
Text: subsequent infestations by the European spruce bark beetle. For Ecol Manage 318:167–174
------------------------------------------------------------
PDF page: 802
Distance: 0.402
Text: as stressed, ash trees can be colonized and killed by this invader (Herms and McCullough 2014). Novel associations between non-native forest insects and other non-host organisms, such as symbiotic fungi, can also lead to severe impacts, particularly by nonnative bark beetles (Wingfield et al. 2017). Many bark beetles (Scolytinae) have important associations with fungi that they introduce into trees. In some tree species, certain mutualistic fungi improve nutrient levels or other conditions for t
------------------------------------------------------------
PDF page: 596
Distance: 0.402
Text: 594 J. Sweeney et al. Fig. 17.4 Examples of signs used to survey for insects: a resin at the sites of southern pine beetle, Dendroctonus frontalis, attack (Photo credit: K. J. Dodds

Tests confirmed that Chroma returns chunk identifiers, PDF page numbers, cosine distances and passage texts. Metadata filtering successfully restricted results to the requested page.

Inspection of the retrieved passages showed mixed relevance. One passage described observable insect attack signs, while other results contained related but less useful information. Retrieval quality therefore requires systematic evaluation rather than relying only on similarity distances.

#### 1.3 Finding the right prompt

The generation step will use a prompt that restricts the language model to evidence retrieved from the textbook.

The prompt requires answers to cite PDF page numbers and to acknowledge when the available passages do not support a reliable answer. The assistant must not invent insect species identifications, pesticide recommendations or predictions about tree survival.

These constraints are intended to reduce unsupported claims and make answers traceable to the source material.

In [62]:
# testing the prompt function
# Retrieving textbook evidence for the question
question = "What signs can indicate a bark beetle attack?"

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

# Build and inspect the grounded prompt
prompt = build_prompt(question, results)

print(prompt[:1200])
print("\n--- END OF PROMPT ---\n")
print(prompt[-400:])


You are the polite and helpful Forest Pest Assistant, helping garden
and small forest owners understand possible insect-related tree problems.

Answer questions using ONLY the textbook sources provided below.

Rules:
- Use only information supported by the provided sources.
- Cite supporting evidence using the exact citation labels supplied with each source.
- Prefer printed textbook pages, for example [Book p. 594].
- If a source has no printed page number, use its PDF page citation.
- Never invent facts, references, or page numbers.
- Do not identify an exact insect species from insufficient symptoms.
- Do not recommend specific pesticides or dosages.
- Do not predict whether a tree will die without sufficient evidence.
- Keep answers clear, polite, and understandable for non-experts.

Clarifying questions:
- If important information is missing, ask a relevant clarifying question.
- Ask only one clarifying question at a time.
- Ask no more than five clarifying questions in total.
- 

The build_prompt() function was implemented in src/functions.py. It combines the user's question with retrieved textbook passages and instructions for evidence-based answers.

The prompt requires PDF page citations, prohibits unsupported diagnoses and pesticide recommendations, and defines how the assistant should respond when evidence is insufficient. It also includes rules for clarifying questions.

The prompt was inspected successfully. Answer generation has not yet been implemented.

#### 1.4 Complete RAG Pipeline

In [55]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load the API key from the project root
load_dotenv(Path("..") / ".env")

# Create the OpenAI client
openai_client = OpenAI()

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("OpenAI client ready:", hasattr(openai_client, "responses"))

API key loaded: True
OpenAI client ready: True


In [3]:
# Load the API key from the project root
load_dotenv(Path("..") / ".env")

# Check that the key is available without displaying it
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

# Create the OpenAI client
client = OpenAI()
print("OpenAI client created")

API key loaded: True
OpenAI client created


In [9]:
# Generate an answer from the grounded prompt
answer = generate_answer(
    prompt=prompt,
    client=client)

print(answer)

Possible signs include resin at attack sites, bark staining, and bark flaking by woodpeckers. These are examples; signs can vary among beetles and trees. [PDF p. 596]


In [54]:
# Test the end-to-end RAG pipeline
question = "What signs can indicate a bark beetle attack?"

answer = ask_tree_assistant(
    question=question,
    model=model,
    collection=collection,
    client=openai_client,
    k=5)

print(answer)

Signs that can be associated with bark beetle attack include **resin at attack sites** and **woodpecker flaking of the bark**. These signs alone do not identify the beetle species or confirm an attack. [Book p. 594]


#### 1.5 Initial testing

In [56]:
# Loading the evaluation dataset from Notebook 01
eval_df = pd.read_csv("../data/evaluation.csv")

# Selecting four questions covering different situations
test_questions = eval_df[eval_df["question_id"].isin(["Q03", "Q06", "Q11", "Q13"])]

# Displaying the selected questions
display(test_questions[["question_id", "question", "answerable"]])

,question_id,question,answerable
2,Q03,What is the difference between insects that eat leaves and leaf miners?,True
5,Q06,My pine tree has small holes in its bark. Could insects be responsible?,True
10,Q11,Can drought make trees more vulnerable to insect attacks?,True
12,Q13,My spruce tree has brown needles. Which exact insect species is attacking it?,False


In [57]:
# Testing the complete RAG pipeline on four evaluation questions
for _, row in test_questions.iterrows():
    question_id = row["question_id"]
    question = row["question"]

    answer = ask_tree_assistant(
        question=question,
        model=model,
        collection=collection,
        client=openai_client,
        k=5)

    print(f"\n{'=' * 60}")
    print(f"Question {question_id}: {question}")
    print(f"Expected answerable: {row['answerable']}")
    print(f"\nAssistant answer:\n{answer}")


Question Q03: What is the difference between insects that eat leaves and leaf miners?
Expected answerable: True

Assistant answer:
Leaf-eating insects feed on the outside of leaves, consuming leaf tissue, sometimes including the veins. Leaf miners feed as larvae between the leaf’s upper and lower surfaces, leaving characteristic patterns of damage. [PDF p. 245]

Question Q06: My pine tree has small holes in its bark. Could insects be responsible?
Expected answerable: True

Assistant answer:
Yes, insects could be involved: the textbook notes that some bark beetles directly damage trees through feeding [Book p. 14]. But it doesn’t say that small holes alone are enough to identify insect damage, so I can’t tell what caused them from that description.

Could you describe how the holes are arranged and whether you see any other changes to the bark?

Question Q11: Can drought make trees more vulnerable to insect attacks?
Expected answerable: True

Assistant answer:
Yes. Drought can weaken s

Verifying the page number metadata for question 3.

In [ ]:
# Inspecting retrieved source pages for Q03
question = test_questions.loc[
    test_questions["question_id"] == "Q03",
    "question"
].iloc[0]

results = search_chroma(
    question=question,
    model=model,
    collection=collection,
    k=5)

# Displaying the citation metadata for each retrieved chunk
for i in range(len(results["ids"][0])):
    print("Chunk:", results["ids"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("-" * 40)

Chunk: page_245_chunk_120
Metadata: {'pdf_page': 245}
----------------------------------------
Chunk: page_245_chunk_0
Metadata: {'pdf_page': 245}
----------------------------------------
Chunk: page_190_chunk_240
Metadata: {'book_page': 182, 'pdf_page': 190}
----------------------------------------
Chunk: page_423_chunk_0
Metadata: {'book_page': 418, 'pdf_page': 423}
----------------------------------------
Chunk: page_190_chunk_360
Metadata: {'pdf_page': 190, 'book_page': 182}
----------------------------------------


In [ ]:
# Inspecting the original header and extracted printed page
page_text = pdf.pages[244].extract_text() or ""

print("First lines of PDF page 245:")
print("\n".join(page_text.splitlines()[:5]))

print("\nDetected book page:", get_book_page(page_text))

First lines of PDF page 245:
Chapter 9 
Foliage Feeders 
Joseph Elkinton and Artemis Roehrig 
9.1 Introduction 
One´ of´ the´ most´ signiﬁcant´ categories´ of´ insects´ that´ cause´ damage´ to´ trees´ are´ the´

Detected book page: None


As no page number was detected the pdf page number is given.

**Summary**

The complete RAG pipeline was tested on four evaluation questions: three answerable questions and one requiring uncertainty.

The assistant produced relevant, textbook-grounded answers with citations. It explained leaf-feeding damage and drought-related insect vulnerability, and it avoided making definitive diagnoses from limited symptoms.

Two limitations were observed. The answer about bark holes relied on relatively general evidence, and the unanswerable species-identification question introduced a specific insect before sufficient symptoms were established. These cases highlight opportunities to improve retrieval relevance and uncertainty handling.

Citation metadata was also verified. Chapter-opening pages without a reliably extracted printed page number correctly use PDF-page citations as a fallback.

---
## 2. LLM track — one design choice, compared

Chunk size, `k`, a metadata filter, the embedding model, the prompt or the tools: change **one** thing and measure it on the evaluation set (at least the retrieval hit rate). Keep the better version, and log the cost of each run.

#### 2.1 Comparison

#### 2.2 The best model

---
## 3. The final version

Which version you will evaluate in notebook 03 and use in the app, and why. Save it: the fitted pipeline with `joblib` (model track), or the index and the configuration (LLM track), in `models/`.

#### 3.1 Final configuration

#### 3.2 Saving the pipeline

#### 3.3 Reloading and testing